# Uji Sensitivitas Konstruksi Shock dan Definisi Forward Linkage

**Skripsi:** Penyusunan Model Simulasi *Spatial Spillover Effects* Sektor Pariwisata Menggunakan *Interregional Input-Output Network* dan *Geospatial Big Data* (Studi Kasus Provinsi Bali Tahun 2025)

**Penulis:** Atikah Nurfadia — 222212522

---

Notebook ini menjawab dua pertanyaan yang diajukan tim penguji pada sidang.

1. Apakah besaran dampak berubah apabila indikator NDVI dan NO₂ tidak di-*clipping*? (pertanyaan no. 5)
2. Mengapa *forward linkage* dihitung dari invers Ghosh dan bukan dari baris matriks Leontief? (pertanyaan no. 13)

Notebook dirancang agar dapat dijalankan ulang dari awal. Bagian 3 dan 4 memuat **verifikasi**, yaitu pemeriksaan bahwa kode di sini menghasilkan angka yang sama persis dengan Tabel 16, Tabel 17, dan Tabel 20 pada naskah. Verifikasi tersebut menjadi dasar kepercayaan terhadap hasil skenario pada bagian 5 dan 6.

**Berkas yang dibutuhkan:** `01_Data_IRIO_17_2025_RAS.xlsx` (keluaran RAS, memuat sheet `A`, `Z`, `FD25`, `Total Output`).

## 0. Pengaturan

In [13]:
import numpy as np
import pandas as pd

from google.colab import drive
drive.mount('/content/drive')

# Sesuaikan path berikut dengan lokasi berkas di komputer atau Google Drive Anda.
PATH_IRIO = '/content/drive/MyDrive/[01] Tugas-Atikah-STIS/[01] Skripsi Atikah/Data/01. Data_IRIO_17_Siap.xlsx'

N_PROV, N_SEK = 34, 17
N = N_PROV * N_SEK          # 578 unit sektor-wilayah
ID_BALI = 17                # Provinsi Bali adalah provinsi ke-17

pd.set_option('display.float_format', lambda v: f'{v:,.4f}')
print('Dimensi sistem:', N, 'unit sektor-wilayah')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dimensi sistem: 578 unit sektor-wilayah


## 1. Data indikator dan pemetaan subsektor

Nilai `|δ|` diambil dari Tabel 15 pada naskah, yaitu koefisien perubahan antara periode *baseline* dan periode *shock* yang dihitung dengan persamaan (37)–(40).

Dua indikator, yaitu NDVI dan NO₂, menghasilkan perubahan yang berlawanan arah dengan yang diharapkan pada Tabel 9. Pada skenario dasar keduanya ditetapkan menjadi nol melalui *clipping* sebelum normalisasi.

In [2]:
# |delta| setiap indikator (Tabel 15), sebelum perlakuan clipping
DELTA_ABS = {
    'NTL':          0.251257,   'LST':          0.071121,
    'NDBI':         0.060517,   'NDWI':         0.036046,
    'BUI':          0.042551,   'NDVI':         0.034174,
    'NO2':          0.032700,   'CO':           0.051233,
    'AER_AI':       0.109634,   'HCHO':         0.274988,
    'Retail':       0.382700,   'Grocery':      0.310600,
    'Parks':        0.559300,   'Transit':      0.669700,
    'Workplaces':   0.206300,   'Residential':  0.144800,
    'RLM':          0.063011,   'TPK':          0.932316,
    'Penerb_ke':    0.837211,   'Penerb_dari':  0.836289,
    'Pesawat_ke':   0.951361,   'Pesawat_dari': 0.940567,
    'Penyeb_ke':    0.790665,   'Kapal_ke':     0.856901,
    'Kapal_dari':   0.823023,
}

# Indikator yang arah perubahannya tidak sesuai ekspektasi (Tabel 18)
KONTRARIAN = ['NDVI', 'NO2']

print(f'Jumlah indikator: {len(DELTA_ABS)}')
print(f'Indikator kontrarian: {KONTRARIAN}')

Jumlah indikator: 25
Indikator kontrarian: ['NDVI', 'NO2']


In [3]:
# Komposisi indikator setiap subsektor (Tabel 16) dan kategori induknya (Tabel 13)
SUBSEKTOR = {
    'G':       (['NTL','LST','NDBI','BUI','NDVI','Retail'],                                        'G'),
    'H_rel':   (['NTL','NO2','Transit'],                                                           'H'),
    'H_darat': (['NO2','CO','NTL','LST','AER_AI','Transit','Workplaces'],                          'H'),
    'H_laut':  (['Kapal_dari','Kapal_ke','Penyeb_ke','NO2','CO','NTL'],                            'H'),
    'H_udara': (['Pesawat_ke','Pesawat_dari','Penerb_ke','Penerb_dari','NO2','CO','NTL','LST',
                 'HCHO','Transit'],                                                                'H'),
    'H_gd':    (['NTL','NO2','CO','LST','NDBI','AER_AI','Workplaces','Transit'],                   'H'),
    'I_mkn':   (['NTL','LST','NDBI','CO','Retail'],                                                'I'),
    'I_ako':   (['NTL','LST','NDBI','BUI','NDWI','TPK','RLM','Retail'],                            'I'),
    'JK':      (['NTL','LST','NDBI','BUI','Workplaces'],                                           'JK'),
    'L':       (['NTL','LST','NDBI','BUI','NDWI'],                                                 'L'),
    'MN':      (['Workplaces','NTL','LST','NDBI','BUI'],                                           'MN'),
    'P':       (['Residential','Workplaces','NTL','LST','NDBI','NDVI'],                            'P'),
    'Q':       (['Grocery','NTL','LST','NDBI'],                                                    'Q'),
    'RSTU':    (['NTL','LST','NDBI','BUI','NDVI','Retail','Parks'],                                'RSTU'),
}

cek = pd.DataFrame([{'Subsektor': s, 'Kategori': k, 'Jumlah indikator': len(i)}
                    for s, (i, k) in SUBSEKTOR.items()])
print('Pemeriksaan terhadap Tabel 13 (jumlah indikator per subsektor):')
cek

Pemeriksaan terhadap Tabel 13 (jumlah indikator per subsektor):


,Subsektor,Kategori,Jumlah indikator
0,G,G,6
1,H_rel,H,3
2,H_darat,H,7
3,H_laut,H,6
4,H_udara,H,10
5,H_gd,H,8
6,I_mkn,I,5
7,I_ako,I,8
8,JK,JK,5
9,L,L,5


## 2. Fungsi normalisasi dan agregasi

Tahapannya mengikuti persamaan (41)–(44) pada naskah.

1. Normalisasi min-max setiap indikator **di dalam subsektornya**, persamaan (41).
2. Skor subsektor adalah rata-rata nilai ternormalisasi, persamaan (43).
3. Skor kategori adalah rata-rata skor subsektor di bawahnya.
4. Koefisien shock `δᵢ` adalah skor kategori dengan tanda negatif, persamaan (44).

Argumen `perlakuan` mengatur bagaimana indikator kontrarian diperlakukan, sedangkan argumen `lingkup_norm` mengatur apakah min-max dihitung di dalam subsektor atau secara global.

In [4]:
def nilai_indikator(ind, perlakuan):
    """Mengembalikan |delta| satu indikator sesuai perlakuan yang dipilih."""
    if perlakuan == 'clipping'     and ind in KONTRARIAN:      return 0.0
    if perlakuan == 'balik_ndvi'   and ind == 'NO2':           return 0.0
    # 'tanpa_clipping', 'keluarkan', dan NDVI pada 'balik_ndvi' memakai nilai apa adanya
    return DELTA_ABS[ind]


def hitung_skor(perlakuan='clipping', lingkup_norm='subsektor'):
    """
    perlakuan    : 'clipping'      -> NDVI & NO2 ditetapkan nol  (skenario dasar)
                   'tanpa_clipping'-> keduanya memakai |delta| apa adanya
                   'keluarkan'     -> keduanya dikeluarkan dari himpunan indikator
                   'balik_ndvi'    -> polaritas NDVI dibalik (dipakai), NO2 tetap nol
    lingkup_norm : 'subsektor' -> min-max di dalam subsektor  (sesuai naskah)
                   'global'    -> min-max terhadap seluruh indikator
                   'tanpa'     -> tanpa normalisasi, rata-rata |delta| mentah
    """
    # kumpulkan nilai untuk min-max global bila diperlukan
    semua = [nilai_indikator(i, perlakuan)
             for s, (inds, k) in SUBSEKTOR.items() for i in inds
             if not (perlakuan == 'keluarkan' and i in KONTRARIAN)]
    g_lo, g_hi = min(semua), max(semua)

    skor_sub, rinci = {}, []
    for s, (inds, kat) in SUBSEKTOR.items():
        pakai = [i for i in inds if not (perlakuan == 'keluarkan' and i in KONTRARIAN)]
        v = np.array([nilai_indikator(i, perlakuan) for i in pakai])

        if lingkup_norm == 'tanpa':
            norm = v
        else:
            lo, hi = (g_lo, g_hi) if lingkup_norm == 'global' else (v.min(), v.max())
            # persamaan (42): bila max = min, seluruh nilai dinormalisasi menjadi 0
            norm = np.zeros_like(v) if hi == lo else (v - lo) / (hi - lo)

        skor_sub[s] = norm.mean()                      # persamaan (43)
        rinci += [(kat, s, i, a, b) for i, a, b in zip(pakai, v, norm)]

    per_kat = {}
    for s, (_, kat) in SUBSEKTOR.items():
        per_kat.setdefault(kat, []).append(skor_sub[s])
    skor_kat = {k: float(np.mean(v)) for k, v in per_kat.items()}
    delta_i  = {k: -v for k, v in skor_kat.items()}    # persamaan (44)

    rinci = pd.DataFrame(rinci, columns=['Kategori','Subsektor','Indikator','abs_delta','ternormalisasi'])
    return skor_sub, skor_kat, delta_i, rinci

print('Fungsi siap.')

Fungsi siap.


## 3. Verifikasi terhadap Tabel 16 dan Tabel 17

Sebelum skenario dijalankan, kode di atas harus terbukti menghasilkan angka yang sama dengan naskah. Apabila ada satu saja baris yang tidak cocok, hasil skenario pada bagian berikutnya tidak dapat dipercaya.

In [5]:
skor_sub, skor_kat, delta_i, rinci = hitung_skor('clipping', 'subsektor')

# Skor subsektor sebagaimana tercetak pada Tabel 16
REF_SUB = {'G':0.351949, 'H_rel':0.458393, 'H_darat':0.289947, 'H_laut':0.539362,
           'H_udara':0.513341, 'H_gd':0.265000, 'I_mkn':0.338292, 'I_ako':0.216334,
           'JK':0.126349, 'L':0.261383, 'MN':0.401513, 'P':0.486882,
           'Q':0.451277, 'RSTU':0.349275}

v = pd.DataFrame([{'Subsektor': s,
                   'Tabel 16': REF_SUB[s],
                   'Hitung ulang': skor_sub[s],
                   'Selisih': skor_sub[s] - REF_SUB[s],
                   'Status': 'COCOK' if abs(skor_sub[s] - REF_SUB[s]) < 5e-6 else 'TIDAK COCOK'}
                  for s in SUBSEKTOR])
v

,Subsektor,Tabel 16,Hitung ulang,Selisih,Status
0,G,0.3519,0.3519,0.0000,COCOK
1,H_rel,0.4584,0.4584,-0.0000,COCOK
2,H_darat,0.2899,0.2899,0.0000,COCOK
3,H_laut,0.5394,0.5394,-0.0000,COCOK
4,H_udara,0.5133,0.5133,0.0000,COCOK
5,H_gd,0.2650,0.2650,-0.0000,COCOK
6,I_mkn,0.3383,0.3383,-0.0000,COCOK
7,I_ako,0.2163,0.2163,0.0000,COCOK
8,JK,0.1263,0.4015,0.2752,TIDAK COCOK
9,L,0.2614,0.2614,-0.0000,COCOK


In [6]:
print('Rincian subsektor J,K:')
print(rinci[rinci.Subsektor == 'JK'][['Indikator','abs_delta','ternormalisasi']].to_string(index=False))
print(f'\nRata-rata nilai ternormalisasi = {rinci[rinci.Subsektor=="JK"].ternormalisasi.mean():.6f}')

Rincian subsektor J,K:
 Indikator  abs_delta  ternormalisasi
       NTL     0.2513          1.0000
       LST     0.0711          0.1369
      NDBI     0.0605          0.0861
       BUI     0.0426          0.0000
Workplaces     0.2063          0.7846

Rata-rata nilai ternormalisasi = 0.401513


In [8]:
# Skor kategori sebagaimana tercetak pada Tabel 17
REF_KAT = {'G':0.351949, 'H':0.413209, 'I':0.277313, 'JK':0.401513, 'L':0.261383,
           'MN':0.401513, 'P':0.486882, 'Q':0.451277, 'RSTU':0.349275}

v = pd.DataFrame([{'Kategori': k,
                   'Tabel 17': REF_KAT[k],
                   'Hitung ulang': skor_kat[k],
                   'Selisih': skor_kat[k] - REF_KAT[k],
                   'Status': 'COCOK' if abs(skor_kat[k] - REF_KAT[k]) < 5e-6 else 'TIDAK COCOK'}
                  for k in REF_KAT])
print('Seluruh kategori cocok:', (v.Status == 'COCOK').all())
v

Seluruh kategori cocok: True


,Kategori,Tabel 17,Hitung ulang,Selisih,Status
0,G,0.3519,0.3519,0.0000,COCOK
1,H,0.4132,0.4132,-0.0000,COCOK
2,I,0.2773,0.2773,0.0000,COCOK
3,JK,0.4015,0.4015,0.0000,COCOK
4,L,0.2614,0.2614,-0.0000,COCOK
5,MN,0.4015,0.4015,0.0000,COCOK
6,P,0.4869,0.4869,-0.0000,COCOK
7,Q,0.4513,0.4513,0.0000,COCOK
8,RSTU,0.3493,0.3493,-0.0000,COCOK


## 4. Verifikasi terhadap Tabel 20

Koefisien `δᵢ` dipetakan ke sektor IRIO menurut Tabel 19, lalu dikalikan permintaan akhir Provinsi Bali tahun 2025 untuk memperoleh ΔFD. Hasilnya dibandingkan dengan Tabel 20.

In [14]:
import os

A   = pd.read_excel(PATH_IRIO, sheet_name='A',           header=None).values.astype(float)
FD = pd.read_excel(PATH_IRIO, sheet_name='FD25',         header=None).iloc[:, 0].values.astype(float)
Z  = pd.read_excel(PATH_IRIO, sheet_name='Z',            header=None).values.astype(float)
x  = pd.read_excel(PATH_IRIO, sheet_name='Total Output', header=None).iloc[:, 0].values.astype(float)

assert A.shape == (N, N), f'Dimensi A tidak sesuai: {A.shape}'
L = np.linalg.inv(np.eye(N) - A)          # invers Leontief
print(f'A  : {A.shape}   radius spektral = {max(abs(np.linalg.eigvals(A))):.4f}')
print(f'FD : {FD.shape}  total nasional  = Rp{FD.sum()/1e6:,.2f} triliun')

A  : (578, 578)   radius spektral = 0.5947
FD : (578,)  total nasional  = Rp25,795.22 triliun


In [15]:
# Tabel 19: pemetaan kategori KBLI ke sektor IRIO
PETA_SEKTOR = {7:'G', 8:'H', 9:'I', 10:'JK', 11:'JK', 12:'L', 13:'MN', 15:'P', 16:'Q', 17:'RSTU'}
NAMA_SEKTOR = {7:'Perdagangan Besar dan Eceran', 8:'Transportasi dan Pergudangan',
               9:'Penyediaan Akomodasi dan Makan Minum', 10:'Informasi dan Komunikasi',
               11:'Jasa Keuangan dan Asuransi', 12:'Real Estate', 13:'Jasa Perusahaan',
               15:'Jasa Pendidikan', 16:'Jasa Kesehatan dan Kegiatan Sosial', 17:'Jasa Lainnya'}

def idx(sektor, provinsi=ID_BALI):
    """Indeks 0-based unit sektor-wilayah pada vektor 578x1."""
    return (provinsi - 1) * N_SEK + (sektor - 1)

def bentuk_delta_f(delta_i):
    df = np.zeros(N)
    for sek, kat in PETA_SEKTOR.items():
        df[idx(sek)] = delta_i[kat] * FD[idx(sek)]      # persamaan (35)
    return df

delta_f = bentuk_delta_f(delta_i)

REF_T20 = {7:-7.40, 8:-20.28, 9:-27.58, 10:-4.95, 11:-3.20,
           12:-2.57, 13:-0.54, 15:-9.53, 16:-5.76, 17:-2.58}
v = pd.DataFrame([{'Sektor': f'S{s}', 'Nama': NAMA_SEKTOR[s],
                   'FD 2025 (T Rp)': FD[idx(s)]/1e6,
                   'ΔFD hitung (T Rp)': delta_f[idx(s)]/1e6,
                   'ΔFD Tabel 20': REF_T20[s],
                   'Status': 'COCOK' if abs(delta_f[idx(s)]/1e6 - REF_T20[s]) < 0.01 else 'TIDAK COCOK'}
                  for s in PETA_SEKTOR])
print(f'Total shock hitung ulang = Rp{delta_f.sum()/1e6:,.2f} triliun   (Tabel 20: Rp-84,39 triliun)')
print('Seluruh sektor cocok:', (v.Status == 'COCOK').all())
v

Total shock hitung ulang = Rp-84.39 triliun   (Tabel 20: Rp-84,39 triliun)
Seluruh sektor cocok: True


,Sektor,Nama,FD 2025 (T Rp),ΔFD hitung (T Rp),ΔFD Tabel 20,Status
0,S7,Perdagangan Besar dan Eceran,21.0336,-7.4028,-7.4000,COCOK
1,S8,Transportasi dan Pergudangan,49.0841,-20.2820,-20.2800,COCOK
2,S9,Penyediaan Akomodasi dan Makan Minum,99.4366,-27.5751,-27.5800,COCOK
3,S10,Informasi dan Komunikasi,12.3279,-4.9498,-4.9500,COCOK
4,S11,Jasa Keuangan dan Asuransi,7.9660,-3.1984,-3.2000,COCOK
5,S12,Real Estate,9.8465,-2.5737,-2.5700,COCOK
6,S13,Jasa Perusahaan,1.3538,-0.5436,-0.5400,COCOK
7,S15,Jasa Pendidikan,19.5725,-9.5295,-9.5300,COCOK
8,S16,Jasa Kesehatan dan Kegiatan Sosial,12.7529,-5.7551,-5.7600,COCOK
9,S17,Jasa Lainnya,7.3923,-2.5819,-2.5800,COCOK


Seluruh angka cocok. Kode pada notebook ini karena itu mereproduksi rangkaian konstruksi shock dalam skripsi secara utuh, sehingga skenario alternatif pada bagian berikut dapat dipercaya.

## 5. Skenario perlakuan NDVI dan NO₂

Empat skenario dibandingkan.

| Skenario | Perlakuan |
|---|---|
| A | NDVI dan NO₂ ditetapkan nol melalui *clipping* — **skenario dasar yang dipakai dalam skripsi** |
| B | Keduanya dipertahankan dengan `\|δ\|` apa adanya, tanpa *clipping* |
| C | Keduanya dikeluarkan sepenuhnya, sehingga pembagi pada persamaan (43) berkurang |
| D | Polaritas NDVI dibalik dengan logika yang sama seperti indikator *Residential*, sedangkan NO₂ tetap nol |

Skenario D perlu dijelaskan. Indikator *Residential* juga bergerak naik, tetapi kenaikannya tidak di-*clipping* melainkan dibaca sebagai penurunan mobilitas, karena bertambahnya waktu di tempat tinggal memang berarti berkurangnya aktivitas di luar rumah. NDVI dapat diperlakukan serupa, yaitu kenaikan tutupan vegetasi dibaca sebagai berkurangnya tekanan aktivitas antropogenik. NO₂ tidak diperlakukan demikian karena kenaikan NO₂ pada masa pembatasan bertentangan dengan pola yang terdokumentasi secara luas, sehingga lebih mungkin merupakan derau pengukuran daripada sinyal berlawanan arah.

In [16]:
def simulasi(delta_i):
    """Menjalankan dX = L . df dan merangkum hasilnya."""
    df = bentuk_delta_f(delta_i)
    dx = L @ df
    prov = np.repeat(np.arange(1, N_PROV + 1), N_SEK)
    bali, luar = dx[prov == ID_BALI].sum(), dx[prov != ID_BALI].sum()
    return {'Shock (T Rp)':        df.sum() / 1e6,
            'Dampak total (T Rp)': dx.sum() / 1e6,
            'Multiplier':          dx.sum() / df.sum(),
            'Dampak Bali (T Rp)':  bali / 1e6,
            'Spillover (T Rp)':    luar / 1e6,
            'Pangsa spillover (%)': 100 * luar / dx.sum()}

SKENARIO = [
    ('A. Clipping ke nol (dasar)',      'clipping',       'subsektor'),
    ('B. Tanpa clipping',               'tanpa_clipping', 'subsektor'),
    ('C. NDVI & NO2 dikeluarkan',       'keluarkan',      'subsektor'),
    ('D. Polaritas NDVI dibalik',       'balik_ndvi',     'subsektor'),
]

hasil, koef = {}, {}
for nama, perlakuan, lingkup in SKENARIO:
    _, _, di, _ = hitung_skor(perlakuan, lingkup)
    koef[nama]  = {k: -v for k, v in di.items()}
    hasil[nama] = simulasi(di)

tab_koef = pd.DataFrame(koef)
print('Koefisien shock δᵢ per kategori (nilai positif = besarnya kontraksi):')
tab_koef

Koefisien shock δᵢ per kategori (nilai positif = besarnya kontraksi):


,A. Clipping ke nol (dasar),B. Tanpa clipping,C. NDVI & NO2 dikeluarkan,D. Polaritas NDVI dibalik
G,0.3519,0.3047,0.3501,0.3047
H,0.4132,0.3939,0.4399,0.4132
I,0.2773,0.2773,0.2773,0.2773
JK,0.4015,0.4015,0.4015,0.4015
L,0.2614,0.2614,0.2614,0.2614
MN,0.4015,0.4015,0.4015,0.4015
P,0.4869,0.4323,0.4524,0.4323
Q,0.4513,0.4513,0.4513,0.4513
RSTU,0.3493,0.3162,0.3587,0.3162


In [17]:
tab_hasil = pd.DataFrame(hasil).T
dasar = tab_hasil.iloc[0]
tab_hasil['Selisih dampak (T Rp)'] = tab_hasil['Dampak total (T Rp)'] - dasar['Dampak total (T Rp)']
tab_hasil['Selisih dampak (%)']    = 100 * tab_hasil['Selisih dampak (T Rp)'] / abs(dasar['Dampak total (T Rp)'])
tab_hasil.round(4)

,Shock (T Rp),Dampak total (T Rp),Multiplier,Dampak Bali (T Rp),Spillover (T Rp),Pangsa spillover (%),Selisih dampak (T Rp),Selisih dampak (%)
A. Clipping ke nol (dasar),-84.3919,-142.6493,1.6903,-112.4256,-30.2237,21.1874,0.0000,0.0000
B. Tanpa clipping,-81.1395,-137.3540,1.6928,-108.1445,-29.2095,21.2658,5.2953,3.7121
C. NDVI & NO2 dikeluarkan,-85.0544,-144.2609,1.6961,-113.4380,-30.8230,21.3661,-1.6117,-1.1298
D. Polaritas NDVI dibalik,-82.0873,-139.2022,1.6958,-109.4646,-29.7376,21.3629,3.4470,2.4164


### Pembacaan hasil

Besaran dampak memang berubah, tetapi perubahannya terbatas, yaitu antara −1,13 persen dan +3,71 persen terhadap nilai yang dilaporkan sebesar Rp142,65 triliun.

Yang lebih penting, ukuran-ukuran struktural yang menjadi dasar kesimpulan penelitian hampir tidak bergerak. *Multiplier* efektif tetap pada kisaran 1,69 dan pangsa spillover tetap pada kisaran 21 persen pada keempat skenario. Urutan provinsi penerima dan komposisi sektor penyalur spillover juga tidak berubah, karena keduanya ditentukan oleh struktur matriks Leontief dan bukan oleh besaran `δᵢ`.

Perlu dicatat bahwa *clipping* bukan perlakuan yang paling konservatif maupun paling longgar. Skenario dasar menghasilkan shock yang lebih besar daripada skenario B tetapi lebih kecil daripada skenario C, karena *clipping* menempatkan nilai nol sebagai batas bawah normalisasi sekaligus tetap menghitung indikator tersebut sebagai pembagi. Kedua pengaruh itu bekerja berlawanan arah dan sebagian saling meniadakan.

In [18]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
nama_pendek = ['A','B','C','D']

ax[0].bar(nama_pendek, -tab_hasil['Dampak total (T Rp)'], color='#8C1C13')
ax[0].set_ylabel('Dampak output nasional (triliun Rp)')
ax[0].set_title('Besaran dampak menurut skenario')
for i, v in enumerate(-tab_hasil['Dampak total (T Rp)']):
    ax[0].text(i, v + 1.5, f'{v:.1f}'.replace('.', ','), ha='center', fontsize=9)
ax[0].set_ylim(0, 165)

ax[1].bar(nama_pendek, tab_hasil['Pangsa spillover (%)'], color='#0B6E99')
ax[1].set_ylabel('Pangsa spillover (%)')
ax[1].set_title('Pangsa spillover menurut skenario')
for i, v in enumerate(tab_hasil['Pangsa spillover (%)']):
    ax[1].text(i, v + 0.3, f'{v:.2f}'.replace('.', ','), ha='center', fontsize=9)
ax[1].set_ylim(0, 26)

for a in ax:
    a.spines[['top','right']].set_visible(False)
    a.grid(axis='y', alpha=.25, linestyle=':')
plt.tight_layout()
plt.savefig('gambar_sensitivitas_skenario.png', dpi=200, bbox_inches='tight')
plt.show()

## 6. Skenario alternatif lingkup normalisasi

Bagian ini menguji pertanyaan yang berbeda, yaitu seberapa besar hasil bergantung pada keputusan melakukan normalisasi min-max **di dalam subsektor**, bukan pada perlakuan terhadap indikator kontrarian.

| Skenario | Lingkup normalisasi |
|---|---|
| E | min-max dihitung terhadap seluruh 25 indikator sekaligus |
| F | tanpa normalisasi, skor subsektor adalah rata-rata `\|δ\|` mentah |

Hasil bagian ini **tidak dimasukkan sebagai skenario utama** dalam naskah, karena keduanya bukan variasi perlakuan dalam metode yang sama melainkan metode normalisasi yang berbeda. Bagian ini disertakan agar konsekuensinya terdokumentasi dan dapat dijelaskan apabila ditanyakan.

In [19]:
SKENARIO_NORM = [
    ('A. Min-max dalam subsektor (dasar)', 'clipping', 'subsektor'),
    ('E. Min-max global',                  'clipping', 'global'),
    ('F. Tanpa normalisasi',               'clipping', 'tanpa'),
]
hasil_n = {}
for nama, perlakuan, lingkup in SKENARIO_NORM:
    _, _, di, _ = hitung_skor(perlakuan, lingkup)
    hasil_n[nama] = simulasi(di)

t = pd.DataFrame(hasil_n).T
t['Selisih dampak (%)'] = 100 * (t['Dampak total (T Rp)'] - t['Dampak total (T Rp)'].iloc[0]) \
                          / abs(t['Dampak total (T Rp)'].iloc[0])
t.round(4)

,Shock (T Rp),Dampak total (T Rp),Multiplier,Dampak Bali (T Rp),Spillover (T Rp),Pangsa spillover (%),Selisih dampak (%)
A. Min-max dalam subsektor (dasar),-84.3919,-142.6493,1.6903,-112.4256,-30.2237,21.1874,0.0000
E. Min-max global,-50.5322,-88.1111,1.7437,-67.8322,-20.2789,23.0152,38.2323
F. Tanpa normalisasi,-48.0744,-83.8255,1.7437,-64.5329,-19.2926,23.0152,41.2367


In [20]:
# Mengapa normalisasi dalam-subsektor menghasilkan skor yang lebih tinggi
baris = []
for s, (inds, kat) in SUBSEKTOR.items():
    v = np.array([nilai_indikator(i, 'clipping') for i in inds])
    norm = (v - v.min()) / (v.max() - v.min())
    baris.append({'Subsektor': s,
                  'Rata-rata |δ| mentah': v.mean(),
                  'Skor min-max dalam subsektor': norm.mean(),
                  'Rasio': norm.mean() / v.mean(),
                  'Indikator terbesar': inds[int(v.argmax())],
                  '|δ| terbesar': v.max()})
pd.DataFrame(baris).sort_values('Rasio', ascending=False).round(4)

,Subsektor,Rata-rata |δ| mentah,Skor min-max dalam subsektor,Rasio,Indikator terbesar,|δ| terbesar
11,P,0.1223,0.4869,3.9800,NTL,0.2513
8,JK,0.1263,0.4015,3.1778,NTL,0.2513
10,MN,0.1263,0.4015,3.1778,NTL,0.2513
9,L,0.0923,0.2614,2.8319,NTL,0.2513
0,G,0.1347,0.3519,2.6130,Retail,0.3827
12,Q,0.1734,0.4513,2.6029,Grocery,0.3106
6,I_mkn,0.1634,0.3383,2.0708,Retail,0.3827
13,RSTU,0.1953,0.3493,1.7879,Parks,0.5593
1,H_rel,0.3070,0.4584,1.4932,Transit,0.6697
2,H_darat,0.1942,0.2899,1.4932,Transit,0.6697


### Pembacaan hasil

Normalisasi min-max di dalam subsektor menaikkan skor dibandingkan rata-rata `|δ|` mentah, dengan rasio antara 0,94 kali dan 3,98 kali. Penyebabnya adalah sifat min-max itu sendiri, yaitu indikator terbesar dalam setiap subsektor selalu dipetakan menjadi 1 berapa pun besar perubahan sesungguhnya. Pada subsektor Real Estat, misalnya, perubahan NTL sebesar 25 persen menjadi bernilai 1, seolah-olah merepresentasikan kontraksi maksimum.

Pola kenaikannya tidak acak. Rasio terbesar justru terjadi pada subsektor yang seluruh indikatornya berasal dari citra satelit dengan rentang perubahan sempit, yaitu Jasa Pendidikan, Jasa Perusahaan, J K, dan Real Estat. Sebaliknya, rasio terkecil terjadi pada subsektor yang indikatornya didominasi statistik resmi dengan perubahan besar dan nyata, yaitu Angkutan Udara dan Angkutan Laut.

Dengan kata lain, normalisasi dalam subsektor paling banyak menaikkan skor pada subsektor yang dasar pengukurannya paling lemah.

**Pembelaan atas pilihan yang dipakai.** Normalisasi di dalam subsektor tetap dapat dipertahankan dengan alasan bahwa indikator memiliki kepekaan yang berbeda terhadap perubahan aktivitas yang sama. NTL merupakan proksi kasar yang bergerak sedikit meskipun aktivitas turun tajam, sedangkan jumlah penumpang bergerak hampir sebanding dengan aktivitasnya. Normalisasi di dalam kelompok indikator yang mengukur aktivitas yang sama berfungsi menyetarakan kepekaan tersebut. Apabila normalisasi dilakukan secara global, indikator berkepekaan rendah akan selalu memperoleh bobot kecil dan subsektor yang hanya memiliki indikator semacam itu akan tampak nyaris tidak terdampak, padahal aktivitasnya mungkin benar-benar berhenti.

Keterbatasan dari pilihan tersebut dicatat pada subbab 5.2.

## 7. Perbandingan definisi *forward linkage*

Bagian ini menjawab pertanyaan penguji nomor 13. Dua definisi dibandingkan.

- **FL Ghosh**, yaitu jumlah baris invers Ghosh `G = (I − B)⁻¹` dengan `B = x̂⁻¹Z`, sesuai persamaan (54)–(55) pada naskah. Mengukur dorongan dari sisi penawaran.
- **FL baris Leontief**, yaitu jumlah baris `L = (I − A)⁻¹`. Mengukur kepekaan output sektor i terhadap kenaikan permintaan akhir di seluruh sektor, sehingga masih merupakan ukuran sisi permintaan.

Keduanya dinormalisasi terhadap rata-rata seluruh 578 unit sektor-wilayah, sama seperti pada naskah.

In [21]:
x_inv = np.where(x != 0, 1 / x, 0.0)
B = x_inv[:, None] * Z                     # koefisien alokasi Ghosh
G = np.linalg.inv(np.eye(N) - B)           # invers Ghosh

BL   = L.sum(axis=0) / L.sum() * N         # persamaan (53)
FL_g = G.sum(axis=1) / G.sum() * N         # persamaan (55), berbasis Ghosh
FL_l = L.sum(axis=1) / L.sum() * N         # alternatif: baris Leontief

print(f'Korelasi Pearson  FL Ghosh vs FL Leontief : {np.corrcoef(FL_g, FL_l)[0,1]:.4f}')
print(f'Korelasi Spearman FL Ghosh vs FL Leontief : '
      f'{pd.Series(FL_g).corr(pd.Series(FL_l), method="spearman"):.4f}')

Korelasi Pearson  FL Ghosh vs FL Leontief : 0.2830
Korelasi Spearman FL Ghosh vs FL Leontief : 0.6059


In [22]:
NAMA_17 = ['Pertanian','Pertambangan','Industri Pengolahan','Listrik & Gas','Air & Limbah',
           'Konstruksi','Perdagangan','Transportasi','Akomodasi & Mamin','Informasi & Komunikasi',
           'Keuangan & Asuransi','Real Estate','Jasa Perusahaan','Adm. Pemerintahan','Pendidikan',
           'Kesehatan','Jasa Lainnya']

def klasifikasi(bl, fl):
    if bl >= 1 and fl >= 1: return 'Sektor kunci'
    if bl >= 1:             return 'Kuat ke belakang'
    if fl >= 1:             return 'Kuat ke depan'
    return 'Kurang terkait'

sl = slice((ID_BALI - 1) * N_SEK, ID_BALI * N_SEK)
fl_tab = pd.DataFrame({'Kode': [f'S{i+1}' for i in range(N_SEK)], 'Sektor': NAMA_17,
                       'BL': BL[sl], 'FL (Ghosh)': FL_g[sl], 'FL (baris Leontief)': FL_l[sl]})
fl_tab['Klasifikasi (Ghosh)']    = [klasifikasi(r.BL, r._4) for r in fl_tab.itertuples()]
fl_tab['Klasifikasi (Leontief)'] = [klasifikasi(r.BL, r._5) for r in fl_tab.itertuples()]
fl_tab['Berubah'] = np.where(fl_tab['Klasifikasi (Ghosh)'] != fl_tab['Klasifikasi (Leontief)'], 'YA', '')

print(f"Sektor yang berubah klasifikasi: {(fl_tab.Berubah == 'YA').sum()} dari {N_SEK}")
fl_tab.round(3)

Sektor yang berubah klasifikasi: 4 dari 17


,Kode,Sektor,BL,FL (Ghosh),FL (baris Leontief),Klasifikasi (Ghosh),Klasifikasi (Leontief),Berubah
0,S1,Pertanian,0.8300,1.0720,1.0770,Kuat ke depan,Kuat ke depan,
1,S2,Pertambangan,0.9030,1.2380,0.6590,Kuat ke depan,Kurang terkait,YA
2,S3,Industri Pengolahan,1.1360,0.8840,0.8830,Kuat ke belakang,Kuat ke belakang,
3,S4,Listrik & Gas,2.4800,2.1180,1.7750,Sektor kunci,Sektor kunci,
4,S5,Air & Limbah,1.0120,0.8540,0.6080,Kuat ke belakang,Kuat ke belakang,
5,S6,Konstruksi,1.1940,0.6470,0.7200,Kuat ke belakang,Kuat ke belakang,
6,S7,Perdagangan,0.8680,0.9880,0.9270,Kurang terkait,Kurang terkait,
7,S8,Transportasi,1.1530,0.9130,1.2130,Kuat ke belakang,Sektor kunci,YA
8,S9,Akomodasi & Mamin,1.0030,0.7550,1.2620,Kuat ke belakang,Sektor kunci,YA
9,S10,Informasi & Komunikasi,0.9830,1.1370,1.0650,Kuat ke depan,Kuat ke depan,


### Pembacaan hasil

Kedua ukuran tidak dapat saling menggantikan. Korelasi Pearson antara keduanya hanya 0,283 dan korelasi Spearman 0,606, sehingga keduanya memeringkat sektor secara berbeda.

Empat sektor berubah klasifikasi. Yang paling berarti adalah Transportasi dan Pergudangan serta Penyediaan Akomodasi dan Makan Minum, yang akan naik menjadi sektor kunci apabila baris Leontief dipakai. Perubahan itu patut diduga merupakan artefak dari besarnya permintaan akhir kedua sektor tersebut di Bali, yaitu Rp49,08 triliun dan Rp99,44 triliun, bukan cerminan perannya sebagai pemasok input. Sebaliknya, Jasa Perusahaan yang berperan nyata sebagai penyedia jasa antara justru akan turun dari sektor kunci.

Penggantian definisi karena itu akan mengaburkan salah satu temuan penelitian, yaitu bahwa sektor pariwisata inti di Bali tidak menempati posisi sentral dalam jaringan produksi, sehingga besarnya dampak lebih ditentukan oleh besarnya guncangan daripada oleh posisi jaringannya.

## 8. Ekspor hasil

In [23]:
import shutil

# Menentukan path penyimpanan di Google Drive Anda
path_excel_drive = '/content/drive/MyDrive/[01] Tugas-Atikah-STIS/[01] Skripsi Atikah/Data/Output Simulasi Shock/hasil_uji_sensitivitas.xlsx'
path_gambar_drive = '/content/drive/MyDrive/[01] Tugas-Atikah-STIS/[01] Skripsi Atikah/Data/Output Simulasi Shock/gambar_sensitivitas_skenario.png'

# Menyimpan langsung berkas Excel ke Google Drive
with pd.ExcelWriter(path_excel_drive) as w:
    v_sub = pd.DataFrame([{'Subsektor': s, 'Tabel 16': REF_SUB[s], 'Hitung ulang': skor_sub[s]}
                          for s in SUBSEKTOR])
    v_sub.to_excel(w, sheet_name='Verifikasi Tabel 16', index=False)
    tab_koef.to_excel(w, sheet_name='Delta_i per skenario')
    tab_hasil.round(4).to_excel(w, sheet_name='Hasil skenario A-D')
    t.round(4).to_excel(w, sheet_name='Lingkup normalisasi E-F')
    fl_tab.round(4).to_excel(w, sheet_name='FL Ghosh vs Leontief', index=False)

# Menyalin gambar dari direktori lokal ke Google Drive
shutil.copy('gambar_sensitivitas_skenario.png', path_gambar_drive)

print(f'Tersimpan di Drive: {path_excel_drive}')
print(f'Tersimpan di Drive: {path_gambar_drive}')

Tersimpan di Drive: /content/drive/MyDrive/[01] Tugas-Atikah-STIS/[01] Skripsi Atikah/Data/Output Simulasi Shock/hasil_uji_sensitivitas.xlsx
Tersimpan di Drive: /content/drive/MyDrive/[01] Tugas-Atikah-STIS/[01] Skripsi Atikah/Data/Output Simulasi Shock/gambar_sensitivitas_skenario.png


---

## Ringkasan

1. Kode pada notebook ini mereproduksi Tabel 16, Tabel 17, dan Tabel 20 pada naskah secara tepat, kecuali satu salah ketik pada skor subsektor J,K yang seharusnya 0,401513 dan bukan 0,126349.

2. Perlakuan terhadap NDVI dan NO₂ memengaruhi besaran dampak dalam rentang sempit, yaitu antara −1,13 persen dan +3,71 persen. *Multiplier* dan pangsa spillover praktis tidak berubah, sehingga kesimpulan penelitian tahan terhadap pilihan perlakuan ini.

3. Lingkup normalisasi berpengaruh jauh lebih besar daripada perlakuan indikator kontrarian. Hal ini didokumentasikan pada bagian 6 dan dicatat sebagai keterbatasan, bukan dilaporkan sebagai skenario utama, karena merupakan metode normalisasi yang berbeda dan bukan variasi perlakuan dalam metode yang sama.

4. *Forward linkage* berbasis Ghosh dan berbasis baris Leontief memeringkat sektor secara berbeda, dengan korelasi Pearson 0,283. Empat dari tujuh belas sektor Bali berubah klasifikasi, sehingga kedua ukuran tersebut tidak dapat dipertukarkan.